# 04 --- Structured Error Handling

**CCA Pattern**: Subagents return structured error context so the coordinator can retry, flag gaps, or adjust confidence.

**Anti-pattern**: Silent failures return `{"status":"success","data":null}` -- the coordinator can't tell if the source was empty or failed.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.tools.handlers import dispatch
from research_agents.anti_patterns.silent_failures import handle_fetch_page_silent
from tests.conftest import make_services
services = make_services()

## The ToolErrorResponse Model

When a tool handler encounters an error, it returns a `ToolErrorResponse` (from `models/errors.py`):

```python
class ToolErrorResponse(BaseModel):
    status: str = "error"
    error_type: str    # "timeout", "not_found", "rate_limit", etc.
    source: str        # which service/URL failed
    message: str       # human-readable error description
    retry_eligible: bool      # Can the coordinator retry?
    fallback_available: bool   # Is there an alternative source?
    partial_data: dict | None  # Any data recovered before failure
```

This gives the coordinator a **decision tree**:
- `retry_eligible=True` -> retry (timeouts, rate limits)
- `fallback_available=True` -> try alternative source
- Both `False` -> flag gap in the final report

### The Anti-Pattern: SilentFailureResponse

```python
class SilentFailureResponse(BaseModel):
    status: str = "success"  # LIES
    data: None = None         # No data, but claims success
```

The coordinator cannot distinguish between:
- 'no relevant data exists' (legitimate empty result)
- 'the subagent failed to retrieve data' (error needing handling)

## Simulated Data: Intentional Failures

The project's test data (in `data/sources.py`) includes URLs that intentionally fail:

| URL | Behavior | Purpose |
|-----|----------|---------|
| `timeout.example.com/remote-data` | Simulated timeout | Tests retry logic |
| `healthtech.example.com/ai-revolution` | Simulated 404 | Tests fallback logic |

Let's see how each handler responds to these failures.

## Anti-Pattern: Silent Failure

In [ ]:
# Silent failure on timeout -- returns success with null
silent_result = json.loads(handle_fetch_page_silent(
    {'url': 'https://timeout.example.com/remote-data'}, services
))
print('Silent failure response:')
print(json.dumps(silent_result, indent=2))
print(f'\nCan coordinator tell this was a timeout? {"error_type" in silent_result}')

## Correct Pattern: Structured Error Context

In [ ]:
# Structured error on timeout -- coordinator gets full decision tree
structured_result = json.loads(dispatch(
    'web_researcher', 'fetch_page',
    {'url': 'https://timeout.example.com/remote-data'}, services
))
print('Structured error response:')
print(json.dumps(structured_result, indent=2))
print(f'\nCan coordinator retry? {structured_result.get("retry_eligible")}')
print(f'Error type: {structured_result.get("error_type")}')

In [ ]:
# Also show a 404 error (different decision tree)
not_found_result = json.loads(dispatch(
    'web_researcher', 'fetch_page',
    {'url': 'https://healthtech.example.com/ai-revolution'}, services
))
print('404 error response:')
print(json.dumps(not_found_result, indent=2))
print(f'\nRetry eligible: {not_found_result.get("retry_eligible")}')  # False
print(f'Fallback available: {not_found_result.get("fallback_available")}')  # True

In [ ]:
from helpers import compare_results

compare_results(
    {'status': silent_result['status'], 'has_error_type': 'error_type' in silent_result, 'has_retry_eligible': 'retry_eligible' in silent_result, 'coordinator_can_retry': False},
    {'status': structured_result['status'], 'has_error_type': 'error_type' in structured_result, 'has_retry_eligible': 'retry_eligible' in structured_result, 'coordinator_can_retry': structured_result.get('retry_eligible', False)},
)

## CCA Exam Tip

> The silent failure question presents a scenario where a report is missing data.
> - 'increase timeout duration' -- WRONG (addresses symptoms, not root cause)
> - 'add retry logic' -- WRONG (partially helpful but not the core fix)
> - **'require structured error context from subagents' -- CORRECT**
>
> The key insight: the coordinator needs enough information to make a decision (retry, fallback, or flag gap). Silent failures remove that decision-making ability.